In [ ]:
# Install AutoML libraries
# Note: These have many dependencies, installation may take time
!pip install -q pycaret flaml optuna

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed
np.random.seed(42)

## Part 1: Introduction to AutoML

### What is AutoML?

**Automated Machine Learning (AutoML)** automates the end-to-end process of applying ML to real-world problems.

**AutoML handles:**
- ✅ Data preprocessing
- ✅ Feature engineering
- ✅ Model selection
- ✅ Hyperparameter tuning
- ✅ Ensemble methods
- ✅ Model evaluation

**Benefits:**
- ⚡ Faster prototyping
- 🎯 Finds good baselines
- 🔍 Explores many models automatically
- 📊 Automates tedious tasks
- 🎓 Good for learning

**When to Use:**
- Quick baseline needed
- Exploring new datasets
- Limited ML expertise
- Time constraints
- Model comparison

**When NOT to Use:**
- Highly specialized problems
- Need full control
- Production-critical systems (without review)
- Very large datasets (computationally expensive)

## Part 2: PyCaret - Low-Code ML Library

PyCaret is a Python library that wraps several ML libraries for low-code experience.

In [ ]:
# Load classification dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:\n{df['target'].value_counts()}")
df.head()

In [ ]:
from pycaret.classification import *

# Setup PyCaret environment
# This does preprocessing, train/test split, and more
clf_setup = setup(
    data=df,
    target='target',
    session_id=42,
    verbose=False,
    normalize=True,           # Normalize features
    remove_outliers=False,    # Keep outliers for now
    train_size=0.8           # 80-20 split
)

print("Setup complete!")

In [ ]:
# Compare all available models
# This trains and evaluates 10+ models!
print("Comparing models... (this may take a minute)")
best_models = compare_models(n_select=5, sort='AUC')

print("\n✅ Top 5 models identified!")

In [ ]:
# Get the best model
best_model = best_models[0]

print(f"Best model: {best_model}")

# Evaluate the model
print("\nModel evaluation:")
evaluate_model(best_model)

In [ ]:
# Tune hyperparameters
print("Tuning hyperparameters...")
tuned_model = tune_model(
    best_model,
    optimize='AUC',
    n_iter=10  # Number of iterations
)

print("\n✅ Model tuned!")

In [ ]:
# Make predictions
holdout_pred = predict_model(tuned_model)

print("Predictions on holdout set:")
print(holdout_pred.head())

# Get metrics
from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(
    holdout_pred['target'],
    holdout_pred['prediction_label']
))

In [ ]:
# Create ensemble
print("Creating ensemble...")
ensemble_model = ensemble_model(tuned_model, method='Bagging')

print("\n✅ Ensemble created!")

# Final predictions
final_pred = predict_model(ensemble_model)
print(f"\nFinal ensemble performance on holdout:")
print(f"Accuracy: {(final_pred['prediction_label'] == final_pred['target']).mean():.4f}")

In [ ]:
# Save the model
save_model(ensemble_model, 'pycaret_best_model')
print("Model saved as 'pycaret_best_model.pkl'")

# Load it back (to demonstrate)
loaded_model = load_model('pycaret_best_model')
print("Model loaded successfully!")

## Part 3: FLAML - Fast Lightweight AutoML

FLAML is optimized for speed and resource efficiency.

In [ ]:
from flaml import AutoML

# Prepare data
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Initialize FLAML AutoML
automl = AutoML()

# Configure and run
settings = {
    "time_budget": 60,  # seconds
    "metric": 'roc_auc',
    "task": 'classification',
    "log_file_name": 'flaml_experiment.log',
    "seed": 42
}

print("Running FLAML AutoML (60 second budget)...")
automl.fit(X_train, y_train, **settings)

print("\n✅ AutoML complete!")

In [ ]:
# Print best model and configuration
print("Best model found:")
print(f"  Algorithm: {automl.best_estimator}")
print(f"  ROC-AUC: {automl.best_loss:.4f}")
print(f"\nBest hyperparameters:")
for param, value in automl.best_config.items():
    print(f"  {param}: {value}")

In [ ]:
# Evaluate on test set
from sklearn.metrics import roc_auc_score, accuracy_score

y_pred = automl.predict(X_test)
y_pred_proba = automl.predict_proba(X_test)[:, 1]

print("Test set performance:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

In [ ]:
# Visualize feature importance
if hasattr(automl.model.estimator, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': automl.model.estimator.feature_importances_
    }).sort_values('importance', ascending=False).head(10)
    
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['feature'], feature_importance['importance'])
    plt.xlabel('Importance')
    plt.title('Top 10 Feature Importances (FLAML)')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance not available for this model type")

## Part 4: Regression with AutoML

Let's try AutoML on a regression problem.

In [ ]:
# Load regression dataset
diabetes = load_diabetes()
df_reg = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df_reg['target'] = diabetes.target

print(f"Dataset shape: {df_reg.shape}")
print(f"\nTarget statistics:")
print(df_reg['target'].describe())
df_reg.head()

In [ ]:
# PyCaret for regression
from pycaret.regression import *

reg_setup = setup(
    data=df_reg,
    target='target',
    session_id=42,
    verbose=False,
    normalize=True,
    train_size=0.8
)

print("Regression setup complete!")

In [ ]:
# Compare regression models
print("Comparing regression models...")
best_reg_models = compare_models(n_select=3, sort='R2')

print("\n✅ Top 3 regression models identified!")

In [ ]:
# Tune the best model
best_reg = best_reg_models[0]
print(f"Tuning {best_reg}...")

tuned_reg = tune_model(best_reg, optimize='R2', n_iter=10)
print("\n✅ Model tuned!")

In [ ]:
# Evaluate regression model
holdout_reg = predict_model(tuned_reg)

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

print("Regression metrics on holdout:")
print(f"  R² Score: {r2_score(holdout_reg['target'], holdout_reg['prediction_label']):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(holdout_reg['target'], holdout_reg['prediction_label'])):.4f}")
print(f"  MAE: {mean_absolute_error(holdout_reg['target'], holdout_reg['prediction_label']):.4f}")

In [ ]:
# Visualize predictions
plt.figure(figsize=(10, 6))
plt.scatter(
    holdout_reg['target'],
    holdout_reg['prediction_label'],
    alpha=0.6,
    edgecolors='k'
)
plt.plot(
    [holdout_reg['target'].min(), holdout_reg['target'].max()],
    [holdout_reg['target'].min(), holdout_reg['target'].max()],
    'r--',
    lw=2,
    label='Perfect Prediction'
)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs Predicted (Regression AutoML)')
plt.legend()
plt.tight_layout()
plt.show()

## Part 5: Comparing AutoML Platforms

Let's compare different AutoML approaches on the same dataset.

In [ ]:
import time

# Prepare data
X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

results = []

In [ ]:
# 1. FLAML
print("Testing FLAML...")
start = time.time()

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task='classification',
    metric='roc_auc',
    time_budget=30,
    verbose=0
)

flaml_time = time.time() - start
flaml_pred = flaml_automl.predict_proba(X_test)[:, 1]
flaml_score = roc_auc_score(y_test, flaml_pred)

results.append({
    'Platform': 'FLAML',
    'Time (s)': flaml_time,
    'ROC-AUC': flaml_score,
    'Best Model': flaml_automl.best_estimator
})

print(f"✅ FLAML: {flaml_score:.4f} in {flaml_time:.2f}s")

In [ ]:
# 2. Manual baseline (for comparison)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

print("Testing Manual RF baseline...")
start = time.time()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)

manual_time = time.time() - start
manual_pred = rf.predict_proba(X_test_scaled)[:, 1]
manual_score = roc_auc_score(y_test, manual_pred)

results.append({
    'Platform': 'Manual RF',
    'Time (s)': manual_time,
    'ROC-AUC': manual_score,
    'Best Model': 'RandomForest'
})

print(f"✅ Manual RF: {manual_score:.4f} in {manual_time:.2f}s")

In [ ]:
# Compare results
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values('ROC-AUC', ascending=False)

print("\n" + "="*60)
print("AutoML Platform Comparison")
print("="*60)
print(comparison_df.to_string(index=False))
print("="*60)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC-AUC comparison
axes[0].barh(comparison_df['Platform'], comparison_df['ROC-AUC'], color='steelblue')
axes[0].set_xlabel('ROC-AUC Score')
axes[0].set_title('Performance Comparison')
axes[0].set_xlim(0.9, 1.0)

# Time comparison
axes[1].barh(comparison_df['Platform'], comparison_df['Time (s)'], color='coral')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_title('Speed Comparison')

plt.tight_layout()
plt.show()

## Part 6: Best Practices and Tips

In [ ]:
print("""
AutoML Best Practices:

1. ✅ START WITH AUTOML for baselines
   - Get quick results
   - Understand data better
   - Identify promising models

2. ✅ SET REASONABLE TIME BUDGETS
   - Start with 60-300 seconds
   - Increase if needed
   - Balance speed vs performance

3. ✅ VALIDATE RESULTS
   - Check on holdout data
   - Look for overfitting
   - Understand model decisions

4. ✅ INSPECT TOP MODELS
   - Don't just use the best
   - Consider interpretability
   - Check robustness

5. ✅ USE FOR EXPLORATION
   - Try different features
   - Test hypotheses quickly
   - Compare preprocessing steps

6. ❌ DON'T BLINDLY TRUST
   - Review model choices
   - Understand limitations
   - Test edge cases

7. ❌ DON'T SKIP DATA CLEANING
   - AutoML isn't magic
   - Clean data = better results
   - Handle domain-specific issues

Platform Selection Guide:

Use PyCaret when:
  • Need comprehensive pipeline
  • Want visualization tools
  • Prefer low-code approach
  • Building prototypes

Use FLAML when:
  • Speed is critical
  • Limited compute resources
  • Want cost optimization
  • Production deployment

Use H2O when:
  • Very large datasets
  • Need distributed computing
  • Enterprise deployment
  • Java integration

Use Manual ML when:
  • Full control needed
  • Custom architectures
  • Specialized domains
  • Learning purposes
""")

## 🎯 Key Takeaways

1. **AutoML accelerates ML development** - Quick baselines and model exploration
2. **Multiple platforms available** - PyCaret (comprehensive), FLAML (fast), H2O (scalable)
3. **Not a silver bullet** - Still need data understanding and validation
4. **Great for baselines** - Perfect starting point before custom optimization
5. **Time-performance tradeoff** - More time generally = better models
6. **Interpretability matters** - Don't sacrifice understanding for slight accuracy gains

---

## 📝 Practice Exercises

1. **Compare AutoML Platforms**
   - Load a dataset
   - Run PyCaret, FLAML, and manual baseline
   - Compare results and insights

2. **Feature Engineering Impact**
   - Create new features
   - Run AutoML before/after
   - Measure improvement

3. **Time Budget Experiment**
   - Try different time budgets (10s, 60s, 300s)
   - Plot performance vs time
   - Find optimal budget

---

## 🔗 Resources

- [PyCaret Documentation](https://pycaret.org/)
- [FLAML Documentation](https://microsoft.github.io/FLAML/)
- [H2O AutoML Guide](https://docs.h2o.ai/h2o/latest-stable/h2o-docs/automl.html)
- [AutoML Survey Paper](https://arxiv.org/abs/1810.13306)

---

**Next:** [Notebook 5 - End-to-End Low-Code Project](05_end_to_end_project.ipynb)